In [1]:
#!pip install mysql-connector-python

In [3]:
import sqlite3
import mysql.connector

In [9]:
import sqlite3
import mysql.connector

sqlite_conn = sqlite3.connect("database.sqlite")
sqlite_cursor = sqlite_conn.cursor()

mysql_conn = mysql.connector.connect(
    host="127.0.0.1",
    user="userid",
    password="password",
    database="soccerdata"   
)
mysql_cursor = mysql_conn.cursor()


sqlite_cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = sqlite_cursor.fetchall()


for table in tables:
    table_name = table[0]
    if table_name == 'sqlite_sequence':
        continue

   
    drop_sql = f"DROP TABLE IF EXISTS `{table_name}`;"
    mysql_cursor.execute(drop_sql)

    
    sqlite_cursor.execute(f"PRAGMA table_info({table_name})")
    columns = sqlite_cursor.fetchall()

    col_defs = []
    for col in columns:
        col_name = col[1]
        sqlite_type = col[2].upper()

       
        if "INT" in sqlite_type:
            mysql_type = "INT"
        elif "CHAR" in sqlite_type or "TEXT" in sqlite_type:
            
            if col_name.lower() in ['goal', 'shoton', 'shotoff', 'foulcommit', 'card', 'cross', 'corner', 'possession']:
                mysql_type = "TEXT"
            else:
                mysql_type = "VARCHAR(255)"
        elif "DOUBLE" in sqlite_type or "FLOAT" in sqlite_type or "REAL" in sqlite_type:
            mysql_type = "FLOAT"
        elif "DATE" in sqlite_type or "TIME" in sqlite_type:
            mysql_type = "DATETIME"
        else:
            mysql_type = "VARCHAR(255)"

       
        if col[5] == 1:
            col_defs.append(f"`{col_name}` {mysql_type} PRIMARY KEY")
        else:
            col_defs.append(f"`{col_name}` {mysql_type}")

    create_sql = f"CREATE TABLE `{table_name}` ({', '.join(col_defs)});"
    print(f"Creating table `{table_name}` with SQL: {create_sql}")
    mysql_cursor.execute(create_sql)

mysql_conn.commit()
print("✅ All tables created successfully in MySQL!")


sqlite_cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = sqlite_cursor.fetchall()

batch_size = 1000  

for table in tables:
    table_name = table[0]
    if table_name == 'sqlite_sequence':
        continue

    sqlite_cursor.execute(f"SELECT * FROM {table_name}")
    rows = sqlite_cursor.fetchall()

    if len(rows) == 0:
        print(f"❕ Table `{table_name}` has no data, skipping...")
        continue

    num_columns = len(rows[0])
    placeholders = ", ".join(["%s"] * num_columns)
    insert_sql = f"INSERT INTO `{table_name}` VALUES ({placeholders});"

    print(f"📥 Inserting {len(rows)} rows into `{table_name}` in batches...")
    for i in range(0, len(rows), batch_size):
        batch = rows[i:i+batch_size]
        mysql_cursor.executemany(insert_sql, batch)
        mysql_conn.commit()

print("✅ All data inserted into MySQL successfully!")


mysql_cursor.close()
mysql_conn.close()
sqlite_cursor.close()
sqlite_conn.close()


Creating table `Player_Attributes` with SQL: CREATE TABLE `Player_Attributes` (`id` INT PRIMARY KEY, `player_fifa_api_id` INT, `player_api_id` INT, `date` VARCHAR(255), `overall_rating` INT, `potential` INT, `preferred_foot` VARCHAR(255), `attacking_work_rate` VARCHAR(255), `defensive_work_rate` VARCHAR(255), `crossing` INT, `finishing` INT, `heading_accuracy` INT, `short_passing` INT, `volleys` INT, `dribbling` INT, `curve` INT, `free_kick_accuracy` INT, `long_passing` INT, `ball_control` INT, `acceleration` INT, `sprint_speed` INT, `agility` INT, `reactions` INT, `balance` INT, `shot_power` INT, `jumping` INT, `stamina` INT, `strength` INT, `long_shots` INT, `aggression` INT, `interceptions` INT, `positioning` INT, `vision` INT, `penalties` INT, `marking` INT, `standing_tackle` INT, `sliding_tackle` INT, `gk_diving` INT, `gk_handling` INT, `gk_kicking` INT, `gk_positioning` INT, `gk_reflexes` INT);
Creating table `Player` with SQL: CREATE TABLE `Player` (`id` INT PRIMARY KEY, `player